In [ ]:
import json
import pandas as pd
from groq import Groq  # Assuming you are using Groq based on your previous code
from dotenv import load_dotenv
load_dotenv()
import os
api_key=os.getenv("api_key")
# Initialize your client (make sure your api_key is loaded)
client = Groq(api_key=api_key)

# 1. Load your result data into a DataFrame
with open(r"..\testing\result.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

# --- Define Judge Prompt Templates ---

RELEVANCE_PROMPT_TEMPLATE = """
You are an objective evaluation judge. Your task is to evaluate the ANSWER RELEVANCE of an AI-generated response based on the user's question.

Evaluation Criteria:
- ANSWER RELEVANCE: Determine whether the response directly answers the user's question.
- The response should stay focused on the question without unnecessary tangents or dodging.
- Penalize answers that fail to address the core of the prompt.

Provide your evaluation in this exact format:
Score: <1 to 5 integer>
Reason: <brief explanation>

---
User Question:
{question}

Assistant Response:
{response}
"""
def evaluate_relevance(question, response):
    prompt = RELEVANCE_PROMPT_TEMPLATE.format(question=question, response=response)
    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"


# --- Run Evaluation Across the DataFrame ---

print("--- Running Groundedness & Relevance Judges ---")

relevance_results = []

for index, row in df.iterrows():
    print(f"Evaluating row {index + 1} (ID: {row['id']})...")
    

    
    # Run Relevance Judge
    r_result = evaluate_relevance(row['question'], row['Assistant_Response'])
    relevance_results.append(r_result)

df['Relevance_Evaluation'] = relevance_results

# Display the updated dataframe with evaluations
pd.set_option("display.max_colwidth", None)
display(df[['id', 'question', 'Assistant_Response', 'Relevance_Evaluation']])